In [1]:
import os
import json
import re
from datetime import datetime
from typing import List, Dict, Optional, Literal

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# --- Pydantic模型定义，用于确保输出格式的稳定 ---
class Evidence(BaseModel):
    """从文本中提取的单条证据。"""
    snippet: str = Field(description="The exact, original sentence or phrase from the document.")
    factor_type: Literal["Aggravating Factor", "Mitigating Factor"] = Field(description="The classification of the evidence.")
    severity_score: int = Field(description="The impact score for this specific snippet (0-10).", ge=0, le=10)
    reasoning: str = Field(description="A brief justification for the severity score, referencing the provided principles and examples.")

class EvidenceBasedReport(BaseModel):
    """LLM的结构化输出，包含整体分析和所有找到的证据列表。"""
    holistic_analysis: str = Field(description="A comprehensive analytical paragraph that synthesizes the overall connection between the event and the company's risks, forming the Chain of Thought.")
    evidence_list: List[Evidence] = Field(description="A list of all evidence snippets found in the text that support the holistic analysis.")

# --- 用于从外部文件加载配置的辅助函数 ---
def load_risk_events(filepath: str = "./Data/LLM/risk_events.json") -> List[Dict]:
    """从JSON文件加载风险事件。"""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"错误：加载风险事件文件 '{filepath}' 失败: {e}")
        return []

def load_prompt_template(filepath: str = "./Data/LLM/prompt_template.txt") -> str:
    """从文本文件加载Prompt模板。"""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError as e:
        print(f"错误：加载Prompt模板文件 '{filepath}' 失败: {e}")
        return ""

def parse_filename(filepath: str) -> Dict[str, Optional[str]]:
    """
    解析像 'AAPL_2022-10-28.txt' 这样的文件名以提取公司和日期。
    """
    try:
        basename = os.path.basename(filepath)
        match = re.match(r"([A-Z]+)_(\d{4}-\d{2}-\d{2})\.txt", basename)
        if match:
            company_name = match.group(1)
            filing_date = match.group(2)
            datetime.strptime(filing_date, "%Y-%m-%d") # 验证日期格式
            return {"company_name": company_name, "filing_date": filing_date}
    except (ValueError, IndexError):
        pass
    print(f"警告：无法从文件名 '{filepath}' 中解析公司和日期。将使用默认值。")
    return {"company_name": "Unknown", "filing_date": "Unknown"}


def run_full_analysis(document_filepath: str):
    """
    主执行函数，现在接收文档文件路径作为参数。
    """
    load_dotenv()  # 加载环境变量
    # 检查API密钥是否已设置为环境变量
    if not os.getenv("OPENAI_API_KEY"):
        print("错误：OPENAI_API_KEY 未设置。请在运行脚本前将其设置为环境变量。")
        return

    # --- 加载所有配置 ---
    custom_risk_events = load_risk_events()
    prompt_template_string = load_prompt_template()
    file_info = parse_filename(document_filepath)
    
    if not custom_risk_events or not prompt_template_string:
        print("由于配置或模板文件缺失，无法继续。")
        return

    # --- 初始化模型、解析器和Prompt ---
    llm = ChatOpenAI(model="gpt-4o", temperature=0.0, model_kwargs={"response_format": {"type": "json_object"}})
    parser = JsonOutputParser(pydantic_object=EvidenceBasedReport)
    
    prompt = PromptTemplate(
        template=prompt_template_string,
        input_variables=["document_text", "company_name", "filing_date", "event_name", "event_timeframe", "event_description"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
    
    chain = prompt | llm | parser

    try:
        with open(document_filepath, "r", encoding="utf-8") as f:
            document_content = f.read()
    except FileNotFoundError:
        print(f"错误：在 '{document_filepath}' 未找到文档文件。")
        return

    # --- 循环处理每个事件并运行分析 ---
    all_final_reports = []
    print(f"--- 开始为文档 '{document_filepath}' 进行分析 ---")
    print(f"--- 公司: {file_info['company_name']}, 财报日期: {file_info['filing_date']} ---")
    
    for i, event in enumerate(custom_risk_events, 1):
        print(f"\n({i}/{len(custom_risk_events)}) 正在分析事件: {event['event_name']}...")
        try:
            # LLM提取证据
            evidence_report = chain.invoke({
                "document_text": document_content,
                "company_name": file_info["company_name"],
                "filing_date": file_info["filing_date"],
                "event_name": event["event_name"],
                "event_timeframe": event["event_timeframe"],
                "event_description": event["event_description"]
            })
            
            # Python端计算最终分数
            aggravating_score = sum(e['severity_score'] for e in evidence_report['evidence_list'] if e['factor_type'] == 'Aggravating Factor')
            mitigating_score = sum(e['severity_score'] for e in evidence_report['evidence_list'] if e['factor_type'] == 'Mitigating Factor')
            final_score = max(0, min(100, aggravating_score - mitigating_score))

            # 存储所有信息以供最终报告使用
            all_final_reports.append({
                "event_info": event,
                "holistic_analysis": evidence_report['holistic_analysis'],
                "evidence_list": evidence_report['evidence_list'],
                "calculated_score": final_score
            })
            print(f"  > 分析完成. 找到 {len(evidence_report['evidence_list'])} 条证据. 计算得分为: {final_score}")

        except Exception as e:
            print(f"  > 在分析此事件时发生错误: {e}")
    
    # 打印最终的摘要报告
    print_summary_report(all_final_reports, file_info)


def print_summary_report(reports: List[Dict], file_info: Dict):
    """
    用于打印详细摘要报告的辅助函数。
    """
    if not reports:
        print("\n无分析结果可显示。")
        return

    print("\n\n======================================================")
    print(f"  为 {file_info['company_name']} ({file_info['filing_date']}) 生成的摘要报告")
    print("======================================================")
    
    sorted_reports = sorted(reports, key=lambda x: x.get('calculated_score', 0), reverse=True)

    for report in sorted_reports:
        event = report['event_info']
        score = report['calculated_score']
        holistic_analysis = report['holistic_analysis']
        evidence_list = report['evidence_list']

        print(f"\n--- 事件: {event.get('event_name', 'N/A')} ---")
        print(f"  最终计算得分 (0-100): {score}")
        print(f"\n  整体分析 (思维链):")
        print(f"    {holistic_analysis}")
        
        if evidence_list:
            print("\n  证据分解:")
            for ev in evidence_list:
                sign = "+" if ev['factor_type'] == 'Aggravating Factor' else "-"
                print(f"    [{sign}{ev['severity_score']}/10] \"{ev['snippet']}\"")
                print(f"      理由: {ev['reasoning']}")
        else:
            print("\n  证据分解: 未找到相关证据。")
            
    print("\n======================================================")


if __name__ == "__main__":
    run_full_analysis(document_filepath='AAPL_2022-10-28.txt')


错误：OPENAI_API_KEY 未设置。请在运行脚本前将其设置为环境变量。
